In [5]:
!pip install fairlearn

import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from fairlearn.metrics import MetricFrame, selection_rate
from sklearn.metrics import precision_score, recall_score

# 1. Load data
df = fetch_openml(data_id=1590, as_frame=True).frame

# 2. Prepare data
y = (df["class"] == ">50K").astype(int)
X = pd.get_dummies(df.drop(["class", "race"], axis=1), drop_first=True)
group = df["race"]

# 3. Train-test split
X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X, y, group, test_size=0.3, random_state=42
)

# 4. Train model
model = LogisticRegression(max_iter=2000)
pred = model.fit(X_tr, y_tr).predict(X_te)

# ---- FAIRNESS METRICS FOR FAIRLEARN 0.13 ----

# Equal Opportunity (TPR difference)
eo = MetricFrame(
    metrics=recall_score,
    y_true=y_te,
    y_pred=pred,
    sensitive_features=g_te
).difference()

# Disparate Impact (selection rate ratio)
di = MetricFrame(
    metrics=selection_rate,
    y_true=y_te,
    y_pred=pred,
    sensitive_features=g_te
).ratio()

# Predictive Parity (PPV difference)
pp = MetricFrame(
    metrics=precision_score,
    y_true=y_te,
    y_pred=pred,
    sensitive_features=g_te
).difference()

print("FAIRNESS METRICS")
print("--------------------")
print(f"Equal Opportunity Difference: {eo:.3f}")
print(f"Disparate Impact Ratio:       {di:.3f}")
print(f"Predictive Parity Difference:  {pp:.3f}")

print("\nInterpretation: EO → 0, DI → 1, PP → 0 is fair.")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


FAIRNESS METRICS
--------------------
Equal Opportunity Difference: 0.207
Disparate Impact Ratio:       0.241
Predictive Parity Difference:  0.166

Interpretation: EO → 0, DI → 1, PP → 0 is fair.
